In [1]:
!git clone https://github.com/ultralytics/yolov5  # clone repo
%cd yolov5
%pip install -qr requirements.txt  # install dependencies

import torch
from IPython.display import Image, clear_output  # to display images

clear_output()
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

Setup complete. Using torch 1.9.0+cu102 (Tesla T4)


In [ ]:
!pip install ffmpeg-python

In [2]:
%cd /content
!curl -L "https://app.roboflow.com/ds/gpa16WpFka?key=io0nPXjLzJ" > roboflow.zip; unzip roboflow.zip; rm roboflow.zip
%cat data.yaml

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   888  100   888    0     0   1264      0 --:--:-- --:--:-- --:--:--  1263
100 57.1M  100 57.1M    0     0  25.9M      0  0:00:02  0:00:02 --:--:-- 47.8M
Archive:  roboflow.zip
 extracting: README.dataset.txt      
 extracting: README.roboflow.txt     
 extracting: data.yaml               
   creating: train/
   creating: train/images/
 extracting: train/images/102_jpg.rf.2ac69a32edac3f581efeaff7489f4c7c.jpg  
 extracting: train/images/103_jpg.rf.b57f7b46633acf4335df39de7fe00731.jpg  
 extracting: train/images/106_jpg.rf.d320b0dae92e2df3c6db3734ddfbf94a.jpg  
 extracting: train/images/107_jpg.rf.c9f76415834f7edc732c363de9b0bfc7.jpg  
 extracting: train/images/109_jpg.rf.facd948c072adc4d10da35d26318a53e.jpg  
 extracting: train/images/10_jpg.rf.0e7616463c4d74b26ebc2c1d3e8a2244.jpg  
 extracting: train/images/110_jpg.rf

In [ ]:
# define number of classes based on YAML
import yaml
with open("data.yaml", 'r') as stream:
    num_classes = str(yaml.safe_load(stream)['nc'])

In [ ]:
%cat /content/yolov5/models/yolov5x.yaml

# YOLOv5 🚀 by Ultralytics, GPL-3.0 license

# Parameters
nc: 80  # number of classes
depth_multiple: 1.33  # model depth multiple
width_multiple: 1.25  # layer channel multiple
anchors:
  - [10,13, 16,30, 33,23]  # P3/8
  - [30,61, 62,45, 59,119]  # P4/16
  - [116,90, 156,198, 373,326]  # P5/32

# YOLOv5 backbone
backbone:
  # [from, number, module, args]
  [[-1, 1, Focus, [64, 3]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],  # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],  # 3-P3/8
   [-1, 9, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],  # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
   [-1, 1, SPP, [1024, [5, 9, 13]]],
   [-1, 3, C3, [1024, False]],  # 9
  ]

# YOLOv5 head
head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, C3, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, [1]]

In [ ]:
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def writetemplate(line, cell):
    with open(line, 'w') as f:
        f.write(cell.format(**globals()))

In [ ]:
%%writetemplate /content/yolov5/models/custom_yolov5x.yaml

# parameters
nc: {num_classes}  # number of classes
depth_multiple: 1.33  # model depth multiple
width_multiple: 1.25  # layer channel multiple
anchors:
  - [10,13, 16,30, 33,23]  # P3/8
  - [30,61, 62,45, 59,119]  # P4/16
  - [116,90, 156,198, 373,326]  # P5/32

# YOLOv5 backbone
backbone:
  # [from, number, module, args]
  [[-1, 1, Focus, [64, 3]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],  # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],  # 3-P3/8
   [-1, 9, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],  # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
   [-1, 1, SPP, [1024, [5, 9, 13]]],
   [-1, 3, C3, [1024, False]],  # 9
  ]

# YOLOv5 head
head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, C3, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, [1]],  # cat backbone P3
   [-1, 3, C3, [256, False]],  # 17 (P3/8-small)

   [-1, 1, Conv, [256, 3, 2]],
   [[-1, 14], 1, Concat, [1]],  # cat head P4
   [-1, 3, C3, [512, False]],  # 20 (P4/16-medium)

   [-1, 1, Conv, [512, 3, 2]],
   [[-1, 10], 1, Concat, [1]],  # cat head P5
   [-1, 3, C3, [1024, False]],  # 23 (P5/32-large)

   [[17, 20, 23], 1, Detect, [nc, anchors]],  # Detect(P3, P4, P5)
  ]

In [ ]:
# this is the model configuration we will use for our tutorial 
%cat /content/yolov5/models/custom_yolov5x.yaml


# parameters
nc: 1  # number of classes
depth_multiple: 1.33  # model depth multiple
width_multiple: 1.25  # layer channel multiple
anchors:
  - [10,13, 16,30, 33,23]  # P3/8
  - [30,61, 62,45, 59,119]  # P4/16
  - [116,90, 156,198, 373,326]  # P5/32

# YOLOv5 backbone
backbone:
  # [from, number, module, args]
  [[-1, 1, Focus, [64, 3]],  # 0-P1/2
   [-1, 1, Conv, [128, 3, 2]],  # 1-P2/4
   [-1, 3, C3, [128]],
   [-1, 1, Conv, [256, 3, 2]],  # 3-P3/8
   [-1, 9, C3, [256]],
   [-1, 1, Conv, [512, 3, 2]],  # 5-P4/16
   [-1, 9, C3, [512]],
   [-1, 1, Conv, [1024, 3, 2]],  # 7-P5/32
   [-1, 1, SPP, [1024, [5, 9, 13]]],
   [-1, 3, C3, [1024, False]],  # 9
  ]

# YOLOv5 head
head:
  [[-1, 1, Conv, [512, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 6], 1, Concat, [1]],  # cat backbone P4
   [-1, 3, C3, [512, False]],  # 13

   [-1, 1, Conv, [256, 1, 1]],
   [-1, 1, nn.Upsample, [None, 2, 'nearest']],
   [[-1, 4], 1, Concat, [1]],  # cat backbone P3
   [-1, 3, C3, [256, Fa

In [1]:
# train yolov5s on custom data
# time its performance
%%time
%cd /content/yolov5/
!python train.py --img 640 --batch-size 16 --epochs 200 --data '../data.yaml' --cfg ./models/custom_yolov5x.yaml --weights '' --name yolov5x_results  --cache

[Errno 2] No such file or directory: '/content/yolov5/'
/content
python3: can't open file 'train.py': [Errno 2] No such file or directory
CPU times: user 12.3 ms, sys: 9.55 ms, total: 21.8 ms
Wall time: 131 ms


In [ ]:
!zip -r /content/yolov5x_results.zip /content/yolov5/runs/train/yolov5x_results5

  adding: content/yolov5/runs/train/yolov5x_results5/ (stored 0%)
  adding: content/yolov5/runs/train/yolov5x_results5/hyp.yaml (deflated 44%)
  adding: content/yolov5/runs/train/yolov5x_results5/val_batch2_labels.jpg (deflated 2%)
  adding: content/yolov5/runs/train/yolov5x_results5/weights/ (stored 0%)
  adding: content/yolov5/runs/train/yolov5x_results5/weights/best.pt (deflated 10%)
  adding: content/yolov5/runs/train/yolov5x_results5/weights/last.pt (deflated 10%)
  adding: content/yolov5/runs/train/yolov5x_results5/val_batch0_labels.jpg (deflated 3%)
  adding: content/yolov5/runs/train/yolov5x_results5/results.csv (deflated 83%)
  adding: content/yolov5/runs/train/yolov5x_results5/val_batch1_labels.jpg (deflated 2%)
  adding: content/yolov5/runs/train/yolov5x_results5/val_batch0_pred.jpg (deflated 3%)
  adding: content/yolov5/runs/train/yolov5x_results5/confusion_matrix.png (deflated 36%)
  adding: content/yolov5/runs/train/yolov5x_results5/val_batch2_pred.jpg (deflated 2%)
  add

In [ ]:
from google.colab import files
import pandas as pd
result.to_csv('example_file.csv')
files.download('example_file.csv')

FileNotFoundError: ignored

In [ ]:
import os, ffmpeg

def video_compress(video_full_path, output_file_name, target_size):
    
    min_audio_bitrate = 32000
    max_audio_bitrate = 256000

    probe = ffmpeg.probe(video_full_path)
    # Video duration (s)
    duration = float(probe['format']['duration'])
    # Audio bitrate (bps)
    print(probe['streams'][1]['bit_rate'])
    
    audio_bitrate = float(probe['streams'][1]['bit_rate'])
    # Target total bitrate (bps)
    target_total_bitrate = (target_size * 1024 * 8) / (1.073741824 * duration)
    # Target audio bitrate (bps)
    if 10 * audio_bitrate > target_total_bitrate:
        audio_bitrate = target_total_bitrate / 10
        if audio_bitrate < min_audio_bitrate < target_total_bitrate:
            audio_bitrate = min_audio_bitrate
        elif audio_bitrate > max_audio_bitrate:
            audio_bitrate = max_audio_bitrate

    # Target video bitrate, in bps.
    print(target_total_bitrate)
    print(audio_bitrate)
    print(target_size)
    video_bitrate = target_total_bitrate - audio_bitrate

    i = ffmpeg.input(video_full_path)
    ffmpeg.output(i, os.devnull,
                  **{'c:v': 'libx264', 'b:v': video_bitrate, 'pass': 1, 'f': 'mp4'}
                  ).overwrite_output().run()
    ffmpeg.output(i, output_file_name,
                  **{'c:v': 'libx264', 'b:v': video_bitrate, 'pass': 2, 'c:a': 'aac', 'b:a': audio_bitrate}
                  ).overwrite_output().run()

video_compress('/content/yangin.mp4','output.mp4', 200)

32219
105649.71779039412
32000
200


In [ ]:
!python detect.py --weights /content/best.pt --img 640 --conf 0.25 --source /content/eX6arBgEktw0h3rv.mp4

detect: weights=['/content/best.pt'], source=/content/eX6arBgEktw0h3rv.mp4, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, tfl_int8=False
YOLOv5 🚀 v5.0-375-gd1182c4 torch 1.9.0+cu102 CPU

Fusing layers... 
Model Summary: 476 layers, 87205423 parameters, 0 gradients, 217.1 GFLOPs
video 1/1 (1/552) /content/eX6arBgEktw0h3rv.mp4: 640x384 1 smoke, Done. (2.331s)
video 1/1 (2/552) /content/eX6arBgEktw0h3rv.mp4: 640x384 Done. (2.189s)
video 1/1 (3/552) /content/eX6arBgEktw0h3rv.mp4: 640x384 Done. (2.178s)
video 1/1 (4/552) /content/eX6arBgEktw0h3rv.mp4: 640x384 1 fire, Done. (2.174s)
video 1/1 (5/552) /content/eX6arBgEktw0h3rv.mp4: 640x384 Done. (2.168s)
video 1/1 (6/552) /content/eX6arBgEktw0h3rv